## **Loading the tehran graph map by OSM lib:**
   here we load the Tehran map (the .pbf file is local & we read it  !)

   then we turn it to graph!

In [2]:
from pyrosm import OSM

osm = OSM(r'Tehran_2.osm.pbf') 
nodes, roads = osm.get_network(nodes=True, network_type='driving')
Graph = osm.to_graph(nodes,roads,"networkx")

Graph

## **Applying traffic to the map:**

Now we simulate the traffic by a normal distribution!

details of traffic   :

The traffic's center is at the ENGHELAB square(which is in (35.7013, 51.3939) )

then , for each edge of the graph , we calculate the distance to ENGHELAB square to apply the normal distribution like this:

traffic_intensity = math.exp(-((dist) * * 2) / (2 * sigma * * 2)) + random.uniform(-0.05, 0.05)

**so** ... the traffic is higher near Enghelab . it decreases by distance!  

**Then we classify the traffics by these conditions:**
 
 high traffics: intensity > 0.6  (red dots!)

 medium traffics: intensity > 0.3  (yellow dots!)

 no traffic! : intensity ≤ 0.3  

## **At the end of the cell:**
we visualize the changes to see the traffic simulation!

In [25]:
import random
import math
from ipyleaflet import Map, LayerGroup, CircleMarker

# --- Config ---
enghelab_center = (35.7013, 51.3939)
sigma = 0.037
sample_ratio = 0.99 

print("Applying Gaussian traffic...")

# applying the traffic
for u, v, data in Graph.edges(data=True):
    lat_u, lon_u = Graph.nodes[u]['y'], Graph.nodes[u]['x']
    lat_v, lon_v = Graph.nodes[v]['y'], Graph.nodes[v]['x']
    mid_lat = (lat_u + lat_v) / 2
    mid_lon = (lon_u + lon_v) / 2
    dist = math.hypot(mid_lat - enghelab_center[0], mid_lon - enghelab_center[1])
    base = math.exp(-dist**2 / (2 * sigma**2))
    noise = random.gauss(0, 0.13)
    intensity = max(0, min(1, base + noise))
    if intensity > 0.68: data["traffic"] = "high"
    elif intensity > 0.38: data["traffic"] = "medium"
    else: data["traffic"] = "low"

# visualize
center = (35.6892, 51.3890)
traffic_map = Map(center=center, zoom=11)
traffic_layer = LayerGroup(name="Traffic (Red=High, Orange=Medium)")

count = 0
for u, v, data in Graph.edges(data=True):
    if data.get("traffic") in ["high", "medium"] and random.random() > sample_ratio:
        mid_lat = (Graph.nodes[u]['y'] + Graph.nodes[v]['y']) / 2
        mid_lon = (Graph.nodes[u]['x'] + Graph.nodes[v]['x']) / 2
        color = "red" if data["traffic"] == "high" else "orange"
        rad = 6 if data["traffic"] == "high" else 4
        traffic_layer.add_layer(CircleMarker(
            location=(mid_lat, mid_lon),
            radius=rad,
            color=color,
            fill_color=color,
            fill_opacity=0.35,
            opacity=0.4
        ))
        count += 1

print(f"Added {count} traffic markers (1% sample).")
traffic_map.add_layer(traffic_layer)
print("Map ready! Toggle layer.")
traffic_map

Applying Gaussian traffic...
Added 1019 traffic markers (1% sample).
Map ready! Toggle layer.


Map(center=[35.6892, 51.389], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

## **Blocking Zeynoddin highway to Daneshgah street (by denying the access to the road!):**

zeynoddin highway to daneshgah street id : 3657522493

In [16]:
def block_zeinoddin(graph, node_id=3657522493):
    """Block all edges connected to Zeinoddin node."""
    if node_id in graph:
        print(f"Zeinoddin St. (node {node_id}) blocked.")
        for nbr, edges in graph[node_id].items():
            for k, d in edges.items():
                d['access'] = 'no'
                d['traffic'] = 'blocked'
    else:
        print(f"Node {node_id} not found!")

block_zeinoddin(Graph)

Zeinoddin St. (node 3657522493) blocked.


## **Interactive Marker Setup**

This cell creates an interactive map with **draggable markers** for origin and destination:

- **Red home icon**: Start point (initially near Enghelab Square)  
- **Blue parking icon**: Destination (initially university parking)  
- `draggable=True` allows users to **move markers freely**  


In [17]:
from ipyleaflet import Map, Marker, AwesomeIcon, LayerGroup
import osmnx as ox

# --- Config: Start & End ---
home_coord = (35.7014, 51.3917)      # Home (Enghelab area)
parking_coord = (35.7511, 51.5663)   # University Parking

# Create interactive map
center = (35.6892, 51.3890)
route_map = Map(center=center, zoom=11)

# Icons
home_icon = AwesomeIcon(name='home', marker_color='darkred', icon_color='white')
dest_icon = AwesomeIcon(name='parking', marker_color='darkblue', icon_color='white')

# Draggable markers
start_marker = Marker(
    location=home_coord,
    draggable=True,
    icon=home_icon,
    title='Origin'
)

end_marker = Marker(
    location=parking_coord,
    draggable=True,
    icon=dest_icon,
    title='Destination'
)

# Add to map
route_map.add_layer(start_marker)
route_map.add_layer(end_marker)

# Global line holder
path_line = None

#Display
print("Interactive map ready! Drag markers to update start/end.")
route_map

Interactive map ready! Drag markers to update start/end.


Map(center=[35.6892, 51.389], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

## **Map Markers to Graph Nodes:**

This cell converts **dragged marker locations** to the **nearest road network nodes**:

1. Extract all Marker layers from the map  
2. For each marker:
   - Read title (Origin or Destination)
   - Get current (lat, lon)
   - Use ox.nearest_nodes(G, X=lon, Y=lat) → **closest node in OSM graph**
3. Store results in:
   - origin_node: Start node ID
   - dest_node: End node ID


In [21]:
# --- Extract nearest nodes from markers ---
marker_layers = [layer for layer in route_map.layers if isinstance(layer, Marker)]

origin_node = None
dest_node = None

for m in marker_layers:
    lat, lon = m.location
    nearest = ox.nearest_nodes(Graph, X=lon, Y=lat) 
    
    if m.title == 'Origin':
        origin_node = nearest
    elif m.title == 'Destination':
        dest_node = nearest

print(f"Origin Node: {origin_node}")
print(f"Destination Node: {dest_node}")

Origin Node: 3046782469
Destination Node: 1862259349


## **DFS pathfinding :**

This cell implements **DFS** **exactly as required**:

- **No external pathfinding modules** (pure Python)
- **Respects access='no'** (Zeinoddin blocked)
- **Stack-based LIFO exploration**
- **Path reconstruction via backtracking**
- **Visited set prevents revisiting nodes**


In [26]:
def dfs_path(graph, start, goal):
    """Pure DFS: finds any path from start to goal, respects blocked edges."""
    if start not in graph or goal not in graph:
        return None

    stack = [(start, [start])]  # (current_node, path_so_far)
    visited = set()

    while stack:
        node, path = stack.pop()  # LIFO

        if node in visited:
            continue
        visited.add(node)

        if node == goal:
            return path  # Found!

        for neighbor in graph.neighbors(node):
            edges = graph.get_edge_data(node, neighbor)
            if not edges:
                continue

            # Check if ANY edge is accessible
            accessible = False
            for edata in edges.values():
                if edata.get('access', 'yes') != 'no':
                    accessible = True
                    break

            if not accessible:
                continue  # All edges blocked

            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor]))

    return None  # No path exists

dfs_route = dfs_path(Graph, origin_node, dest_node)

if dfs_route:
    print(f"DFS Path: FOUND ({len(dfs_route)} nodes)")
else:
    print("DFS: No path found (check markers or blocked roads)")

if dfs_route:
    total_distance = 0
    for u, v in zip(dfs_route[:-1], dfs_route[1:]):
        edges = Graph.get_edge_data(u, v)
        if edges:
            # Choose shortest edge between u and v
            min_len = min(edata.get('length', 0) for edata in edges.values())
            total_distance += min_len
    print(f"DFS Real Distance: {total_distance/1000:.3f} km")
else:
    print("No path to measure.")



DFS Path: FOUND (36143 nodes)
DFS Real Distance: 1553.754 km


## **DFS visualization:**

In [27]:
from ipyleaflet import Map, Polyline, LayerGroup, LayersControl

# Create map
dfs_map = Map(center=(35.6892, 51.3890), zoom=11)

if dfs_route:
    # Extract coordinates
    coords = [(Graph.nodes[n]['y'], Graph.nodes[n]['x']) for n in dfs_route if n in Graph.nodes]
    
    # Draw path
    dfs_line = Polyline(
        locations=coords,
        color="red",
        weight=6,
        opacity=0.85,
        name="DFS Path"
    )
    dfs_map.add_layer(dfs_line)

dfs_map.add_control(LayersControl())
print("Red = DFS Path")
dfs_map

Red = DFS Path


Map(center=[35.6892, 51.389], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

## 6. **the A * Pathfinding (Traffic-Optimized):**

A **custom A*** implementation that:
- Uses **geodesic heuristic** (km) for admissible guidance
- **Penalizes traffic**:
  - high → ×4
  - medium → ×2
  - low/none → ×1
- **Respects access='no'** (e.g., blocked Zeinoddin St.)
- **Memory-efficient** with heapq priority queue
- **Reconstructs optimal path** via came_from

---



In [ ]:
# Heuristic: Geodesic Distance (km)

from geopy.distance import geodesic
import heapq
import ipyleaflet
from ipyleaflet import Map, Polyline, LayerGroup

def h(node_a, node_b):
    """Geodesic distance (km) between two nodes admissible heuristic."""
    ya, xa = Graph.nodes[node_a]['y'], Graph.nodes[node_a]['x']
    yb, xb = Graph.nodes[node_b]['y'], Graph.nodes[node_b]['x']
    return geodesic((ya, xa), (yb, xb)).km

def a_star_traffic(graph, src, dst):
    """A* with traffic-aware edge weights and access control."""
    if src not in graph or dst not in graph:
        return None

    open_heap = [(0 + h(src, dst), 0, src, [])]  # (f, g, node, path)
    closed = set()
    g_score = {src: 0}

    while open_heap:
        f, g, current, path = heapq.heappop(open_heap)

        if current in closed:
            continue
        closed.add(current)
        path = path + [current]

        if current == dst:
            return path

        for nbr in graph.neighbors(current):
            edges = graph.get_edge_data(current, nbr)
            if not edges:
                continue

            # Skip blocked edges
            if all(e.get('access', 'yes') == 'no' for e in edges.values()):
                continue

            # Use shortest accessible edge
            valid_edges = [e for e in edges.values() if e.get('access', 'yes') != 'no']
            edge = min(valid_edges, key=lambda e: e.get('length', float('inf')))
            base_len = edge['length']
            traffic = edge.get('traffic', 'low')

            # Traffic penalty
            weight = base_len * {
                'high': 4.0,
                'medium': 2.0,
                'low': 1.0,
                'none': 1.0
            }.get(traffic, 1.0)

            tentative_g = g + weight

            if tentative_g < g_score.get(nbr, float('inf')):
                g_score[nbr] = tentative_g
                f_new = tentative_g + h(nbr, dst)
                heapq.heappush(open_heap, (f_new, tentative_g, nbr, path))

    return None  # No path

# Run A*
astar_route = a_star_traffic(Graph, origin_node, dest_node)
print(f"A* Path: {'Found' if astar_route else 'Not found'}")


# Path Length Calculation

def path_length_traffic(graph, route):
    """Calculate total weighted length (with traffic penalty)."""
    if not route or len(route) < 2:
        return 0.0
    total = 0.0
    for u, v in zip(route[:-1], route[1:]):
        edges = graph.get_edge_data(u, v)
        if not edges:
            continue
        edge = min(edges.values(), key=lambda e: e.get('length', float('inf')))
        base = edge['length']
        traffic = edge.get('traffic', 'low')
        penalty = {'high': 4, 'medium': 2}.get(traffic, 1)
        total += base * penalty
    return total

# Compute
if astar_route:
    real_dist = sum(
        min(e['length'] for e in Graph.get_edge_data(u, v).values())
        for u, v in zip(astar_route[:-1], astar_route[1:])
        if Graph.get_edge_data(u, v)
    ) / 1000

    weighted_dist = path_length_traffic(Graph, astar_route) / 1000

    print(f"Real distance: {real_dist:.3f} km")
    print(f"Traffic-weighted cost: {weighted_dist:.3f} km-equivalent")
else:
    print("No path found.")


# visualization
astar_map = Map(center=(35.6892, 51.3890), zoom=11)

if astar_route:
    coords = [(Graph.nodes[n]['y'], Graph.nodes[n]['x']) for n in astar_route]
    line = Polyline(
        locations=coords,
        color="lime",
        weight=6,
        opacity=0.9,
        name="A* (Traffic-Optimized)"
    )
    astar_map.add_layer(line)

astar_map.add_control(ipyleaflet.LayersControl())
print("A* path in lime green")
astar_map